<a href="https://colab.research.google.com/github/prasath25/Hands-on/blob/main/Experiment8_StudentMemoryAgent_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Experiment 8 — Semantic Memory & Vector Stores for Agents (Colab version)
### Theme: "Student Memory Assistant"

The **Google Colab version** of Experiment 8 — avoids local pip/venv/Python-version issues entirely. Colab's Python is fully supported by `faiss-cpu`, `sentence-transformers`, and `transformers`, so every install here is a normal prebuilt wheel.

**Still no API key of any kind.** Both models — the embedder (`sentence-transformers/all-MiniLM-L6-v2`) and the LLM (`Qwen/Qwen2.5-1.5B-Instruct`) — are public HuggingFace models, no login/token needed.

**One difference from the VS Code version — persistence:** the local project saves memory to a `memory_data/` folder that survives between runs of `app.py`. A Colab session's disk is temporary, so by default memory here only persists *within this session* (until you close/reset the runtime). Section 6 below shows the optional one-line way to save it to your Google Drive instead, so it survives across sessions too.

### Before you run
`Runtime → Change runtime type → T4 GPU` recommended (CPU also works, just slower).
Run the cells **top to bottom, in order**.


## 0. Install dependencies (one-time per session)

In [1]:
!pip install -q -U faiss-cpu sentence-transformers transformers accelerate huggingface_hub

import torch
print("GPU available:", torch.cuda.is_available())


GPU available: True


---
## 1. The SemanticMemory class

Same logic as `memory/memory_store.py` in the VS Code project: embed text with a local model, store vectors in FAISS, keep the raw text alongside so a match can be mapped back to its original sentence.


In [2]:
import os, json as _json
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

class SemanticMemory:
    def __init__(self, embedder_dir: str, storage_dir: str):
        self.embedder = SentenceTransformer(embedder_dir)
        self.storage_dir = storage_dir
        os.makedirs(storage_dir, exist_ok=True)
        self._index_path = os.path.join(storage_dir, "memory.index")
        self._texts_path = os.path.join(storage_dir, "memory_texts.json")
        self.texts = []
        self.index = None
        self._load_or_init()

    def _load_or_init(self):
        if os.path.exists(self._index_path) and os.path.exists(self._texts_path):
            self.index = faiss.read_index(self._index_path)
            with open(self._texts_path, "r", encoding="utf-8") as f:
                self.texts = _json.load(f)
        else:
            dim = self.embedder.get_sentence_embedding_dimension()
            self.index = faiss.IndexFlatL2(dim)
            self.texts = []

    def _save(self):
        faiss.write_index(self.index, self._index_path)
        with open(self._texts_path, "w", encoding="utf-8") as f:
            _json.dump(self.texts, f, indent=2)

    def add(self, text: str) -> None:
        text = text.strip()
        if not text:
            return
        embedding = self.embedder.encode([text]).astype("float32")
        self.index.add(embedding)
        self.texts.append(text)
        self._save()

    def search(self, query: str, k: int = 3):
        if self.index.ntotal == 0:
            return []
        k = min(k, self.index.ntotal)
        embedding = self.embedder.encode([query]).astype("float32")
        distances, indices = self.index.search(embedding, k)
        results = []
        for dist, idx in zip(distances[0], indices[0]):
            if idx == -1:
                continue
            results.append({"text": self.texts[idx], "distance": float(dist)})
        return results

    def count(self) -> int:
        return len(self.texts)

print("SemanticMemory class ready.")


SemanticMemory class ready.


---
## 2. Load the models (public, no API key, no login)

Downloads both models directly from the HuggingFace Hub the first time this cell runs.


In [3]:
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

LLM_ID = "Qwen/Qwen2.5-1.5B-Instruct"
EMBEDDER_ID = "sentence-transformers/all-MiniLM-L6-v2"

tokenizer = AutoTokenizer.from_pretrained(LLM_ID)
model = AutoModelForCausalLM.from_pretrained(LLM_ID, torch_dtype="auto", device_map="auto")

text_gen_pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=200,
    do_sample=False,
    return_full_text=False,
)

def llm(prompt: str) -> str:
    return text_gen_pipeline(prompt)[0]["generated_text"].strip()

# SentenceTransformer can load directly by model ID -- no separate download step needed
memory = SemanticMemory(embedder_dir=EMBEDDER_ID, storage_dir="/content/memory_data")

print("Models loaded. Memory currently has", memory.count(), "fact(s).")


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Models loaded. Memory currently has 0 fact(s).


/tmp/ipykernel_777/115695019.py:23: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  dim = self.embedder.get_sentence_embedding_dimension()


---
## 3. Seed a few starter facts (first run only)

Same starter facts as `data/seed_memories.json` in the VS Code project.


In [4]:
SEED_MEMORIES = [
    "The student's name is Alex.",
    "Alex is majoring in Computer Science.",
    "Alex's favorite subject is Machine Learning.",
    "Alex finds Calculus the most difficult subject.",
    "Alex's final exams begin on December 10th.",
    "Alex is part of the robotics club on campus.",
    "Alex prefers studying in the morning rather than at night.",
]

if memory.count() == 0:
    print("Memory is empty -- preloading starter facts...")
    for fact in SEED_MEMORIES:
        memory.add(fact)
    print(f"Preloaded {len(SEED_MEMORIES)} facts.")
else:
    print(f"Memory already has {memory.count()} fact(s), skipping preload.")


Memory is empty -- preloading starter facts...
Preloaded 7 facts.


---
## 4. The answer function

Retrieves the most semantically similar memories, then hands them to the LLM as context alongside the question.


In [5]:
def answer_question(question: str, memory: SemanticMemory, llm) -> str:
    matches = memory.search(question, k=3)
    context = "\n".join(f"- {m['text']}" for m in matches) if matches else "(no relevant memories found)"
    prompt = (
        "You are a helpful assistant for a student. Use the following known "
        "facts about the student if they are relevant to the question -- "
        "ignore any that aren't relevant.\n\n"
        f"Known facts:\n{context}\n\n"
        f"Question: {question}\n"
        "Answer:"
    )
    return llm(prompt)

print("answer_question ready.")


answer_question ready.


---
## 5. Try it out

Ask about a preloaded fact, then teach it something new and ask about that too.


In [6]:
print(answer_question("What is my favorite subject?", memory, llm))


[transformers] The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


My favorite subject is English Literature. Based on the information provided, it can be inferred that Alex's preferences and interests may not necessarily reflect those of the person being asked the question. However, since Alex's favorite subject is Machine Learning and their difficulty with Calculus suggests they might have an interest or aptitude towards STEM subjects, it could be possible that the person asking the question has a different set of interests or academic strengths. The preference for studying in the morning over night also doesn't provide direct insight into what the person's favorite subject might be. Therefore, without more specific information about the person being asked, we cannot definitively state that "English Literature" is the answer based solely on the given facts. 

However, if we were to make an educated guess based on common preferences and general knowledge, one plausible answer would be that the person's favorite subject is likely **English Literature*

In [ ]:
print(answer_question("When do my final exams start?", memory, llm))


In [ ]:
# Teach it something new
memory.add("I am taking 5 classes this semester")
print(f"Got it -- memory now has {memory.count()} fact(s).")


In [ ]:
print(answer_question("How many classes am I taking?", memory, llm))


---
## 5b. Ask your own question / teach your own fact

Edit either line below and re-run as many times as you like.


In [ ]:
# To teach a new fact, uncomment and edit this line:
# memory.add("Your fact here")

your_question = "What subject does Alex find hardest?"  # <-- edit this line
print(answer_question(your_question, memory, llm))


---
## 6. Optional — persist memory across sessions using Google Drive

By default, memory lives at `/content/memory_data`, which is wiped when this Colab runtime resets. To make it survive across sessions, mount your Drive and point `storage_dir` there instead — run this **before** Section 2's model-loading cell if you want it from the start, or just re-run Section 2 with the new path afterward.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_STORAGE_DIR = "/content/drive/MyDrive/Experiment8_StudentMemory"

# Re-point memory at Drive storage (loads existing memory if this path was used before)
memory = SemanticMemory(embedder_dir=EMBEDDER_ID, storage_dir=DRIVE_STORAGE_DIR)
print(f"Memory now persisted to Google Drive. Currently has {memory.count()} fact(s).")


---
## Troubleshooting

| Symptom | Fix |
|---|---|
| `pip install` cell errors | `Runtime → Restart session`, then re-run the install cell once more before anything else |
| Answers ignore a fact you just added | Make sure `memory.add(...)` actually ran (check the printed count) before asking about it |
| Memory is empty again after reopening Colab | Expected with the default `/content/memory_data` path — use Section 6 to persist to Google Drive instead |
| `CUDA out of memory` | `Runtime → Change runtime type → CPU`, then re-run from the top |
| Retrieved facts seem unrelated to the question | `search()` always returns its *closest* matches even if none are a great fit — expect a generic answer rather than a wrong one for very off-topic questions |
